# Estacionalidad mensual — ¿en qué mes conviene comprar?

Hipótesis de partida: los mercados de criptomonedas presentan patrones estacionales repetibles. Si el precio promedio de cierre de un mes dado es consistentemente inferior al de otros meses a lo largo del histórico, ese mes representa históricamente la ventana de entrada más económica.

Output: `recomendacion_mensual.csv` — un registro por cripto con el mes de precio promedio mínimo y el valor de ese promedio.

In [1]:
# 05_recomendacion_mensual.ipynb
# Recomendación de compra por mes para cada criptomoneda basada en estadísticas históricas

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

## 1. Carga del dataset unificado

In [ ]:
# 1. Cargar dataset unificado

ruta_csv = "../datos/procesados/precios_diarios.csv"
print(f"Leyendo dataset desde: {ruta_csv}")
df = pd.read_csv(ruta_csv, encoding="utf-8", sep=';')
df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True)
print(f"  {len(df)} filas cargadas, rango de fechas {df['fecha'].min().date()} -> {df['fecha'].max().date()}")

## 2. Extracción de mes y año

El mes se extrae como entero (1–12) para facilitar la agrupación numérica y el ordenamiento del eje X en los gráficos. El filtro por año ≤ 2026 excluye datos potencialmente erróneos más allá del rango temporal esperado.

In [ ]:
# 2. Agregar columna mes y año

df["año"] = df["fecha"].dt.year
df["mes"] = df["fecha"].dt.month
print(f"Columnas 'año' y 'mes' extraídas. Años presentes: {sorted(df['año'].unique())}")

# Filtrar hasta 2026
filas_antes = len(df)
df = df[df["año"] <= 2026]
print(f"Filtro año <= 2026 aplicado: {filas_antes - len(df)} filas descartadas (quedan {len(df)})")

## 3. Promedio mensual por criptomoneda

El promedio agrupa todos los años disponibles para cada mes. "Enero" para BTC incluye todos los enero desde 2010. Esto suaviza los efectos de años atípicos, aunque también los diluye: un crash extraordinario de enero 2018 cuenta igual que un enero alcista de 2021.

In [ ]:
# 3. Calcular promedio mensual por cripto

print("Agrupando por (cripto_id, mes) y calculando precio promedio de cierre...")
promedio_mensual = df.groupby(["cripto_id", "mes"])["cierre"].mean().reset_index()
print(f"  {len(promedio_mensual)} combinaciones cripto/mes calculadas (5 criptos x 12 meses = 60 esperadas)")

## 4. Identificación del mejor mes

El mejor mes es el de menor precio promedio histórico por cripto. Este criterio asume que la estacionalidad observada persistirá.

**Cuándo esta asunción es débil:** si el patrón mensual está dominado por uno o dos años atípicos en lugar de repetirse de forma consistente, la recomendación tiene poca solidez. Para reforzarla habría que calcular en cuántos años distintos ese mes fue efectivamente el más bajo.

In [ ]:
# 4. Determinar mejor mes para comprar cada cripto
print("Buscando, por criptomoneda, el mes con menor precio promedio histórico...")
mejor_mes = promedio_mensual.loc[
    promedio_mensual.groupby("cripto_id")["cierre"].idxmin()
].reset_index(drop=True)
print("\nMejor mes de compra por criptomoneda:")
print(mejor_mes)

## 5. Visualización de estacionalidad

El subplot 3×2 compara los cinco perfiles en una sola figura. La **diferencia de altura entre barras** es el dato más relevante: si todas son similares, la diferencia entre meses es marginal y la recomendación tiene poca solidez estadística. Una barra roja claramente más baja que las demás indica un patrón robusto.

In [ ]:
# 5. Graficar precios promedio por mes y marcar mejor mes
print("Generando gráfico de estacionalidad mensual (5 subplots, uno por cripto)...")
nombres_meses = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",
                 7:"Jul",8:"Ago",9:"Sep",10:"Oct",11:"Nov",12:"Dic"}

criptos = sorted(df["cripto_id"].unique())
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, cripto in enumerate(criptos):
    ax = axes[i]
    data = promedio_mensual[promedio_mensual["cripto_id"] == cripto].sort_values("mes")
    mes_min = mejor_mes[mejor_mes["cripto_id"] == cripto]["mes"].values[0]
    colores = ["red" if m == mes_min else "steelblue" for m in data["mes"]]
    ax.bar(data["mes"], data["cierre"], color=colores)
    ax.set_title(cripto)
    ax.set_xlabel("Mes")
    ax.set_ylabel("Precio promedio")
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels([nombres_meses[m] for m in range(1, 13)], rotation=45)
    print(f"  {cripto}: mejor mes = {nombres_meses[mes_min]}")

for j in range(len(criptos), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Precio promedio mensual por criptomoneda (mejor mes en rojo)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Persistencia del resultado

In [ ]:
# 6. Guardar resultados
ruta_salida = "../datos/procesados/recomendacion_mensual.csv"
mejor_mes.to_csv(ruta_salida, index=False, encoding="utf-8", sep=';')
print(f"Recomendaciones guardadas en: {ruta_salida} ({os.path.getsize(ruta_salida)} bytes)")
print(mejor_mes)